## Data Preprocessing

In [1]:
import numpy as np
import pandas as pd
import os
import json
from itertools import product

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

def load_and_merge_radiomics(paths_dict):
    dfs = []
    for modality, path in paths_dict.items():
        df = pd.read_csv(path)
        drop_cols = [c for c in ["sex", "age", "modality"] if c in df.columns]
        df = df.drop(columns=drop_cols, errors="ignore").copy()
        rad_cols = [c for c in df.columns if c != "case_id"]
        df = df.rename(columns={c: f"{modality}_{c}" for c in rad_cols})
        dfs.append(df)
    final_df = dfs[0]
    for d in dfs[1:]:
        final_df = pd.merge(final_df, d, on="case_id", how="inner")
    return final_df


def load_image_features(case_id, base_dir="image_features"):
    modalities = ["ax_t1", "ax_t1c", "ax_t2", "ax_t2f"]
    out = {}
    for m in modalities:
        p = os.path.join(base_dir, str(case_id), m, "image.npy")
        try:
            feats = np.load(p)
            if feats.ndim > 1:
                feats = feats.flatten()
            out[m] = feats
        except FileNotFoundError:
            out[m] = None
    return out


def extract_all_image_features(df_cases, base_dir="image_features"):
    all_features, case_ids = [], []
    ref_dim = None
    for case_id in df_cases["case_id"].values:
        img = load_image_features(case_id, base_dir)
        if all(v is not None for v in img.values()):
            combined = np.concatenate([img[m] for m in ["ax_t1", "ax_t1c", "ax_t2", "ax_t2f"]])
            if ref_dim is None:
                ref_dim = combined.shape[0]
            all_features.append(combined)
        else:
            all_features.append(None)
        case_ids.append(case_id)
    if ref_dim is None:
        return None
    filled = [f if f is not None else np.full(ref_dim, np.nan) for f in all_features]
    cols = [f"img_feat_{i}" for i in range(ref_dim)]
    out = pd.DataFrame(filled, columns=cols)
    out.insert(0, "case_id", case_ids)
    return out


def load_reports(json_path):
    """Load raw radiology reports. Handles str and dict formats (dict uses 'finding' only)."""
    with open(json_path, "r") as f:
        data = json.load(f)
    out = {}
    dict_cases = 0
    for case_id, entry in data.items():
        if not isinstance(entry, dict):
            out[str(case_id)] = ""
            continue
        r = entry.get("report", "")
        if isinstance(r, str):
            out[str(case_id)] = r
        elif isinstance(r, dict):
            dict_cases += 1
            finding = r.get("finding") or r.get("findings") or ""
            out[str(case_id)] = finding if isinstance(finding, str) else ""
        else:
            out[str(case_id)] = ""
    if dict_cases:
        print(f"  [{os.path.basename(json_path)}] {dict_cases} dict-format reports; used 'finding' only.")
    return out


LOCATION_KEYWORDS = ['frontal', 'parietal', 'temporal', 'occipital', 
                     'insular', 'insula', 'falx', 'cerebri', 'cranial', 'fossa', 
                     'convexity', 'cerebellopontine', 'tentorial', 'tentorium', 'sphenoid', 
                     'petrous', 'sagittal', 'sinus', 'sellar', 'sella', 'suprasellar', 
                     'parasellar', 'intrasellar', 'turcica', 'pineal', 'ventricle', 
                     'ventricular', 'third', 'ventricle', 'fourth', 'ventricle', 'lateral', 
                     'ventricle', 'basal', 'ganglia', 'corpus', 'callosum', 'subcortical', 
                     'thalamus', 'cerebellar', 'cerebellum', 'brainstem', 'medulla', 
                     'midbrain', 'left', 'right', 'bilateral', 'multiple']


def prepare_clinical_features(csv_path, age_median=None):
    clin = pd.read_csv(csv_path).copy()
    if age_median is None:
        age_median = clin["Age"].median()
    clin["Age"] = clin["Age"].fillna(age_median)

    loc = clin["Tumor Location"].fillna("").str.lower()
    for kw in LOCATION_KEYWORDS:
        clin[f"loc_{kw}"] = loc.str.contains(kw, regex=False).astype(int)
    clin["loc_missing"] = clin["Tumor Location"].isna().astype(int)
    clin["loc_num_regions"] = loc.str.count(",") + (loc.str.len() > 0).astype(int)
    clin = clin.drop(columns=["Tumor Location"])

    cat_cols = ["Sex",
                "Signal Intensity (T1)", "Signal Intensity (T1c)",
                "Signal Intensity (T2)", "Signal Intensity (T2-FLAIR)"]
    cat_cols = [c for c in cat_cols if c in clin.columns]
    clin = pd.get_dummies(clin, columns=cat_cols, dummy_na=True)
    bool_cols = clin.select_dtypes(include=["bool"]).columns
    clin[bool_cols] = clin[bool_cols].astype(int)
    return clin, age_median


def align_clinical(df_, cols):
    for c in cols:
        if c not in df_.columns:
            df_[c] = 0
    return df_[["case_id"] + cols]


train_json = pd.read_json("train.json")
val_json = pd.read_json("val.json")
train_class = pd.DataFrame(train_json.T.Overall_class, columns=["Overall_class"])
valid_class = pd.DataFrame(val_json.T.Overall_class, columns=["Overall_class"])

le = LabelEncoder()
y_train = le.fit_transform(train_class["Overall_class"])
y_valid = le.transform(valid_class["Overall_class"])
n_classes = len(le.classes_)

print("Class index mapping:")
for i, cls in enumerate(le.classes_):
    print(f"  {i}: {cls}")

class_counts = np.bincount(y_train)
min_class = class_counts.min()
smote_k = max(1, min(5, min_class - 1))
print(f"\nTraining counts: {dict(zip(le.classes_, class_counts))}")
print(f"SMOTE k_neighbors={smote_k}")

print("\nLoading reports...")
reports_train_dict = load_reports("train.json")
reports_val_dict   = load_reports("val.json")
reports_test_dict  = load_reports("test.json")
print(f"Reports: train={len(reports_train_dict)}, val={len(reports_val_dict)}, test={len(reports_test_dict)}")

train_paths = {m: f"{m}_radiomics_train.csv" for m in ["ax_t1", "ax_t1c", "ax_t2", "ax_t2f"]}
val_paths   = {m: f"{m}_radiomics_val.csv"   for m in ["ax_t1", "ax_t1c", "ax_t2", "ax_t2f"]}
test_paths  = {m: f"{m}_radiomics_test.csv"  for m in ["ax_t1", "ax_t1c", "ax_t2", "ax_t2f"]}

rad_train = load_and_merge_radiomics(train_paths)
rad_val   = load_and_merge_radiomics(val_paths)
rad_test  = load_and_merge_radiomics(test_paths)

clin_train, age_med = prepare_clinical_features("train_patient_info.csv")
clin_val,   _       = prepare_clinical_features("val_patient_info.csv",  age_median=age_med)
clin_test,  _       = prepare_clinical_features("test_patient_info.csv", age_median=age_med)
train_clin_cols = [c for c in clin_train.columns if c != "case_id"]
clin_val  = align_clinical(clin_val,  train_clin_cols)
clin_test = align_clinical(clin_test, train_clin_cols)

print(f"\nClinical features: {len(train_clin_cols)}")

print("Extracting image features...")
img_train = extract_all_image_features(clin_train)
img_val   = extract_all_image_features(clin_val)
img_test  = extract_all_image_features(clin_test)


def merge_all(clin_df, rad_df, img_df):
    df = pd.merge(clin_df, rad_df, on="case_id", how="inner")
    if img_df is not None:
        df = pd.merge(df, img_df, on="case_id", how="left")
    return df

df_train = merge_all(clin_train, rad_train, img_train)
df_val   = merge_all(clin_val,   rad_val,   img_val)
df_test  = merge_all(clin_test,  rad_test,  img_test)

clinical_cols  = list(train_clin_cols)
img_cols       = [c for c in df_train.columns if c.startswith("img_feat_")]
radiomics_cols = [c for c in df_train.columns
                  if c not in clinical_cols + img_cols + ["case_id"]]

def align(df, cols):
    df = df.copy()
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
    return df[["case_id"] + cols]

all_cols = clinical_cols + radiomics_cols + img_cols
df_val  = align(df_val,  all_cols)
df_test = align(df_test, all_cols)

print(f"Blocks -> clinical: {len(clinical_cols)}, radiomics: {len(radiomics_cols)}, image: {len(img_cols)}")

train_reports = [reports_train_dict.get(str(cid), "") for cid in df_train["case_id"].values]
val_reports   = [reports_val_dict.get(str(cid),   "") for cid in df_val["case_id"].values]
test_reports  = [reports_test_dict.get(str(cid),  "") for cid in df_test["case_id"].values]

print("\nFitting TF-IDF...")
tfidf = TfidfVectorizer(
    max_features=1500,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    lowercase=True,
    stop_words="english",
    sublinear_tf=True,
)
Xt_tr = tfidf.fit_transform(train_reports).toarray().astype(np.float32)
Xt_va = tfidf.transform(val_reports).toarray().astype(np.float32)
Xt_te = tfidf.transform(test_reports).toarray().astype(np.float32)
print(f"TF-IDF shape: {Xt_tr.shape}")

def impute_block(train_block, *others):
    imp = SimpleImputer(strategy="median")
    tr = imp.fit_transform(train_block)
    ot = [imp.transform(b) for b in others]
    return tr, ot

Xc_tr, (Xc_va, Xc_te) = impute_block(
    df_train[clinical_cols].values, df_val[clinical_cols].values, df_test[clinical_cols].values)

Xr_tr_raw, (Xr_va_raw, Xr_te_raw) = impute_block(
    df_train[radiomics_cols].values, df_val[radiomics_cols].values, df_test[radiomics_cols].values)
rad_scaler = StandardScaler()
Xr_tr_s = rad_scaler.fit_transform(Xr_tr_raw)
Xr_va_s = rad_scaler.transform(Xr_va_raw)
Xr_te_s = rad_scaler.transform(Xr_te_raw)
K_RAD_MAX = min(300, Xr_tr_s.shape[1])
selector = SelectKBest(score_func=f_classif, k=K_RAD_MAX)
Xr_tr_sel = selector.fit_transform(Xr_tr_s, y_train)
Xr_va_sel = selector.transform(Xr_va_s)
Xr_te_sel = selector.transform(Xr_te_s)
rad_rank = np.argsort(selector.scores_[selector.get_support()])[::-1]

img_missing_train = df_train[img_cols].isna().all(axis=1).astype(int).values.reshape(-1, 1)
img_missing_val   = df_val[img_cols].isna().all(axis=1).astype(int).values.reshape(-1, 1)
img_missing_test  = df_test[img_cols].isna().all(axis=1).astype(int).values.reshape(-1, 1)

Xi_tr_raw, (Xi_va_raw, Xi_te_raw) = impute_block(
    df_train[img_cols].values, df_val[img_cols].values, df_test[img_cols].values)
img_scaler = StandardScaler()
Xi_tr_s = img_scaler.fit_transform(Xi_tr_raw)
Xi_va_s = img_scaler.transform(Xi_va_raw)
Xi_te_s = img_scaler.transform(Xi_te_raw)
N_COMP_MAX = min(256, Xi_tr_s.shape[1], Xi_tr_s.shape[0] - 1)
pca_img = PCA(n_components=N_COMP_MAX, random_state=42)
Xi_tr_pca = pca_img.fit_transform(Xi_tr_s)
Xi_va_pca = pca_img.transform(Xi_va_s)
Xi_te_pca = pca_img.transform(Xi_te_s)
print(f"Image PCA: {N_COMP_MAX} dims (EV {pca_img.explained_variance_ratio_.sum():.2%})")


K_RAD = min(200, K_RAD_MAX)
N_IMG = 64

rad_idx = rad_rank[:K_RAD]
X_tr = np.hstack([Xc_tr, Xr_tr_sel[:, rad_idx], Xi_tr_pca[:, :N_IMG], Xt_tr, img_missing_train])
X_va = np.hstack([Xc_va, Xr_va_sel[:, rad_idx], Xi_va_pca[:, :N_IMG], Xt_va, img_missing_val])
X_te = np.hstack([Xc_te, Xr_te_sel[:, rad_idx], Xi_te_pca[:, :N_IMG], Xt_te, img_missing_test])

print(f"Full feature matrix: train {X_tr.shape}, val {X_va.shape}, test {X_te.shape}")


Class index mapping:
  0: Brain Metastase Tumour
  1: Glioma
  2: Meningioma
  3: Pineal tumour and Choroid plexus tumour
  4: Tumors of the sellar region

Training counts: {'Brain Metastase Tumour': np.int64(252), 'Glioma': np.int64(924), 'Meningioma': np.int64(728), 'Pineal tumour and Choroid plexus tumour': np.int64(23), 'Tumors of the sellar region': np.int64(56)}
SMOTE k_neighbors=5

Loading reports...
  [test.json] 11 dict-format reports; used 'finding' only.
Reports: train=1983, val=283, test=378

Clinical features: 77
Extracting image features...
Blocks -> clinical: 77, radiomics: 20, image: 8192

Fitting TF-IDF...
TF-IDF shape: (1983, 909)
Image PCA: 256 dims (EV 86.10%)
Full feature matrix: train (1983, 1071), val (283, 1071), test (378, 1071)


## XGBoost

In [8]:

print("\n" + "="*60)
print("MODEL 1: XGBoost")
print("="*60)

xgb_param_dist = {
    "xgb__max_depth": [3, 4, 5, 6],
    "xgb__learning_rate": [0.03, 0.05, 0.07, 0.1],
    "xgb__n_estimators": [400, 600, 900],
    "xgb__subsample": [0.7, 0.8, 1.0],
    "xgb__colsample_bytree": [0.6, 0.8, 1.0],
    "xgb__min_child_weight": [1, 3, 5],
    "xgb__gamma": [0, 0.3, 0.5],
    "xgb__reg_alpha": [0, 0.001, 0.01, 0.1],
    "xgb__reg_lambda": [0.5, 1.0, 2.0, 5.0],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_pipe = ImbPipeline([
    ("smote", SMOTE(random_state=42, k_neighbors=smote_k)),
    ("xgb", XGBClassifier(
        objective="multi:softprob", num_class=n_classes,
        random_state=42, eval_metric="mlogloss",
        tree_method="hist", device="cuda", n_jobs=1,
    )),
])
xgb_search = RandomizedSearchCV(xgb_pipe, xgb_param_dist, n_iter=20,
                                scoring="f1_macro", cv=cv, n_jobs=1,
                                verbose=0, random_state=42)
xgb_search.fit(X_tr, y_train)
print(f"XGB best CV f1_macro: {xgb_search.best_score_:.4f}")
xgb_best_params = {k.replace("xgb__", ""): v for k, v in xgb_search.best_params_.items()}
print(f"XGB best params: {xgb_best_params}")

smote_final = SMOTE(random_state=42, k_neighbors=smote_k)
X_res, y_res = smote_final.fit_resample(X_tr, y_train)
xgb_final = XGBClassifier(
    objective="multi:softprob", num_class=n_classes,
    random_state=42, eval_metric="mlogloss",
    tree_method="hist", device="cuda",
    early_stopping_rounds=30, **xgb_best_params,
)
xgb_final.fit(X_res, y_res, eval_set=[(X_va, y_valid)], verbose=False)
xgb_val_proba  = xgb_final.predict_proba(X_va)
xgb_test_proba = xgb_final.predict_proba(X_te)
xgb_val_f1 = f1_score(y_valid, xgb_val_proba.argmax(axis=1), average="macro")
print(f"XGB val f1_macro: {xgb_val_f1:.4f}")



MODEL 1: XGBoost


/home/ubuntu/.venv/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [17:30:51] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


XGB best CV f1_macro: 0.7111
XGB best params: {'subsample': 0.8, 'reg_lambda': 0.5, 'reg_alpha': 0.01, 'n_estimators': 600, 'min_child_weight': 3, 'max_depth': 5, 'learning_rate': 0.1, 'gamma': 0, 'colsample_bytree': 0.6}
XGB val f1_macro: 0.7313


## LightGBM

In [ ]:
import warnings

# Filter only this specific warning
warnings.filterwarnings('ignore', message="X does not have valid feature names")

# LightGBM hyperparameter search
lgb_param_dist = {
    "lgbm__num_leaves": [31, 63, 127, 255],
    "lgbm__learning_rate": [0.02, 0.05, 0.07, 0.1],
    "lgbm__n_estimators": [400, 600, 900, 1200],
    "lgbm__subsample": [0.6, 0.7, 0.8, 1.0],
    "lgbm__colsample_bytree": [0.6, 0.7, 0.8, 1.0],
    "lgbm__min_child_samples": [5, 10, 20, 30],
    "lgbm__reg_alpha": [0, 0.001, 0.01, 0.1],
    "lgbm__reg_lambda": [0, 0.5, 1.0, 2.0],
}

lgb_pipe = ImbPipeline([
    ("lgbm", LGBMClassifier(
        objective="multiclass",
        num_class=n_classes,
        random_state=42,
        verbose=-1,
        n_jobs=4,
    )),
])

lgb_search = RandomizedSearchCV(
    lgb_pipe, 
    lgb_param_dist, 
    n_iter=20,  # More iterations for LGBM
    scoring="f1_macro", 
    cv=cv, 
    n_jobs=1,
    verbose=0, 
    random_state=42
)
lgb_search.fit(X_tr, y_train)
print(f"LGBM best CV f1_macro: {lgb_search.best_score_:.4f}")
lgb_best_params = {k.replace("lgbm__", ""): v for k, v in lgb_search.best_params_.items()}
print(f"LGBM best params: {lgb_best_params}")

# Train final LGBM with best params
lgb_final = LGBMClassifier(
    objective="multiclass",
    num_class=n_classes,
    random_state=42,
    verbose=-1,
    n_jobs=4,
    verbosity=-1,
    **lgb_best_params,
)
lgb_final.fit(X_res, y_res, eval_set=[(X_va, y_valid)])
lgb_val_proba  = lgb_final.predict_proba(X_va)
lgb_test_proba = lgb_final.predict_proba(X_te)
lgb_val_f1 = f1_score(y_valid, lgb_val_proba.argmax(axis=1), average="macro")
print(f"LGB val f1_macro: {lgb_val_f1:.4f}")



### Results of Tuning (save time)

In [9]:

print("\n" + "="*60)
print("MODEL 2: LightGBM")
print("="*60)

# using tuned params from grid search
lgb_final = LGBMClassifier(
    objective="multiclass",
    num_class=n_classes,
    num_leaves=63,
    learning_rate=0.02,
    n_estimators=1200,
    subsample=0.6,
    colsample_bytree=0.6,
    min_child_samples=5,
    reg_alpha=0.001,
    reg_lambda=2.0,
    random_state=42,
    verbose=-1,
    n_jobs=4,
)
lgb_final.fit(X_res, y_res, eval_set=[(X_va, y_valid)])
lgb_val_proba  = lgb_final.predict_proba(X_va)
lgb_test_proba = lgb_final.predict_proba(X_te)
lgb_val_f1 = f1_score(y_valid, lgb_val_proba.argmax(axis=1), average="macro")
print(f"LGB val f1_macro: {lgb_val_f1:.4f}")



MODEL 2: LightGBM
LGB val f1_macro: 0.7800


/home/ubuntu/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/ubuntu/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Logistic Regression

In [7]:
print("\n" + "="*60)
print("MODEL 3: Logistic Regression with Hyperparameter Tuning")
print("="*60)

logreg_param_dist = {
    "C": [0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
    "solver": ["lbfgs"], 
    "class_weight": [None, "balanced"],
    "max_iter": [2000, 3000, 4000],
    "tol": [1e-4, 1e-3, 1e-2],
}

# Prepare data
smote_text = SMOTE(random_state=42, k_neighbors=smote_k)
Xt_tr_res, yt_res = smote_text.fit_resample(Xt_tr, y_train)

logreg_search = RandomizedSearchCV(
    LogisticRegression(random_state=42),
    logreg_param_dist,
    n_iter=15,
    scoring="f1_macro",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    verbose=1,
    random_state=42
)

logreg_search.fit(Xt_tr_res, yt_res)
print(f"LogReg best CV f1_macro: {logreg_search.best_score_:.4f}")
print(f"LogReg best params: {logreg_search.best_params_}")

logreg_final = LogisticRegression(
    random_state=42,
    **logreg_search.best_params_
)
logreg_final.fit(Xt_tr_res, yt_res)

# Evaluate
logreg_val_proba = logreg_final.predict_proba(Xt_va)
logreg_test_proba = logreg_final.predict_proba(Xt_te)
logreg_val_f1 = f1_score(y_valid, logreg_val_proba.argmax(axis=1), average="macro")
print(f"LogReg val f1_macro: {logreg_val_f1:.4f}")


MODEL 3: Logistic Regression with Hyperparameter Tuning
Fitting 5 folds for each of 15 candidates, totalling 75 fits
LogReg best CV f1_macro: 0.8993
LogReg best params: {'tol': 0.0001, 'solver': 'lbfgs', 'max_iter': 4000, 'class_weight': None, 'C': 10.0}
LogReg val f1_macro: 0.6484


In [12]:

print("\n" + "="*60)
print("ENSEMBLE: searching weights")
print("="*60)

best_weights = None
best_ens_f1 = -np.inf
weight_grid = [0.0, 0.25, 0.5, 0.75, 1.0]
for w_xgb, w_lgb, w_lr in product(weight_grid, repeat=3):
    total = w_xgb + w_lgb + w_lr
    if total == 0:
        continue
    wx, wl, wr = w_xgb/total, w_lgb/total, w_lr/total
    ens = wx * xgb_val_proba + wl * lgb_val_proba + wr * logreg_val_proba
    pred = ens.argmax(axis=1)
    f1 = f1_score(y_valid, pred, average="macro")
    if f1 > best_ens_f1:
        best_ens_f1 = f1
        best_weights = (wx, wl, wr)

print(f"Best weights (XGB, LGB, LR): "
      f"({best_weights[0]:.2f}, {best_weights[1]:.2f}, {best_weights[2]:.2f})")
print(f"Ensemble val f1_macro: {best_ens_f1:.4f}")

w_xgb, w_lgb, w_lr = best_weights
ens_val_proba  = w_xgb * xgb_val_proba  + w_lgb * lgb_val_proba  + w_lr * logreg_val_proba
ens_test_proba = w_xgb * xgb_test_proba + w_lgb * lgb_test_proba + w_lr * logreg_test_proba

print("\n" + "="*60)
print("THRESHOLD TUNING")
print("="*60)

def predict_with_thresholds(proba, thresholds):
    adjusted = proba / thresholds
    return adjusted.argmax(axis=1)

best_thresholds = np.ones(n_classes)
best_th_f1 = f1_score(y_valid, ens_val_proba.argmax(axis=1), average="macro")
print(f"Baseline (thresholds=1.0): {best_th_f1:.4f}")

threshold_options = [0.3, 0.5, 0.7, 1.0, 1.3, 1.7, 2.0]
for i in range(n_classes):
    best_for_i = best_thresholds[i]
    for t in threshold_options:
        trial = best_thresholds.copy()
        trial[i] = t
        pred = predict_with_thresholds(ens_val_proba, trial)
        f1 = f1_score(y_valid, pred, average="macro")
        if f1 > best_th_f1:
            best_th_f1 = f1
            best_for_i = t
    best_thresholds[i] = best_for_i

for _ in range(2):
    for i in range(n_classes):
        for t in threshold_options:
            trial = best_thresholds.copy()
            trial[i] = t
            pred = predict_with_thresholds(ens_val_proba, trial)
            f1 = f1_score(y_valid, pred, average="macro")
            if f1 > best_th_f1:
                best_th_f1 = f1
                best_thresholds[i] = t

print(f"Tuned thresholds: {dict(zip(le.classes_, best_thresholds.round(2)))}")
print(f"Post-threshold val f1_macro: {best_th_f1:.4f}")

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(f"XGBoost alone        : {xgb_val_f1:.4f}")
print(f"LightGBM alone       : {lgb_val_f1:.4f}")
print(f"LogReg (TF-IDF) alone: {logreg_val_f1:.4f}")
print(f"Ensemble (argmax)    : {best_ens_f1:.4f}")
print(f"Ensemble + thresholds: {best_th_f1:.4f}")

y_val_pred = predict_with_thresholds(ens_val_proba, best_thresholds)
print(f"\nValidation f1_weighted: {f1_score(y_valid, y_val_pred, average='weighted'):.4f}")
print("\nClassification Report (Validation, final model):")
print(classification_report(y_valid, y_val_pred, target_names=le.classes_))
print("Confusion Matrix:")
print(pd.DataFrame(confusion_matrix(y_valid, y_val_pred),
                   index=le.classes_, columns=le.classes_))

y_test_pred = predict_with_thresholds(ens_test_proba, best_thresholds)
y_test_labels = le.inverse_transform(y_test_pred)

submission = pd.DataFrame({
    "case_id": df_test["case_id"].values,
    "Overall_class": y_test_labels,
})
submission.to_csv("tuned_submission.csv", index=False)
print("\nSaved tuned_submission.csv")


ENSEMBLE: searching weights
Best weights (XGB, LGB, LR): (0.33, 0.67, 0.00)
Ensemble val f1_macro: 0.7926

THRESHOLD TUNING
Baseline (thresholds=1.0): 0.7926
Tuned thresholds: {'Brain Metastase Tumour': np.float64(0.3), 'Glioma': np.float64(1.3), 'Meningioma': np.float64(0.3), 'Pineal tumour and Choroid plexus tumour': np.float64(1.0), 'Tumors of the sellar region': np.float64(1.0)}
Post-threshold val f1_macro: 0.8350

FINAL RESULTS
XGBoost alone        : 0.7313
LightGBM alone       : 0.7800
LogReg (TF-IDF) alone: 0.6484
Ensemble (argmax)    : 0.7926
Ensemble + thresholds: 0.8350

Validation f1_weighted: 0.8547

Classification Report (Validation, final model):
                                         precision    recall  f1-score   support

                 Brain Metastase Tumour       0.66      0.69      0.68        36
                                 Glioma       0.90      0.82      0.86       132
                             Meningioma       0.86      0.96      0.91       104
Pinea